# কোথায় চালাবেন

এই notebook দুই জায়গাতেই চলবে:

- **VS Code:** ফাইলটি খুলে প্রতিটি code cell-এর Run button চাপুন, অথবা `Run All` ব্যবহার করুন।
- **Google Colab:** `File > Upload notebook` দিয়ে এই `.ipynb` ফাইল upload করে `Runtime > Run all` চাপুন।

কোনো বাইরের image, file path বা upload দরকার নেই। প্রথম code cell নিজে থেকে 256 x 256 grayscale ও color sample image তৈরি করে। সব পরের section সেই generated sample-এর উপর কাজ করে।

# Easy Digital Image Processing Project

This notebook completes the requested exercises using images generated automatically inside the notebook. Run the cells from top to bottom in Jupyter, VS Code, or Google Colab.

**Input:** The setup cell creates grayscale and RGB sample arrays automatically. No external image, upload, or file path is used.

**Output:** Each section prints the relevant values and displays a clearly labelled figure showing the transformation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter

# This cell creates the only input images used by the whole notebook.
h, w = 256, 256
x = np.linspace(0, 255, w, dtype=np.float32)
gray = np.tile(x, (h, 1)).astype(np.uint8)
pil = Image.fromarray(gray)
draw = ImageDraw.Draw(pil)
draw.ellipse((40, 55, 145, 160), fill=220)
draw.rectangle((155, 75, 225, 180), fill=40)
gray = np.array(pil)
color = np.stack([gray, np.roll(gray, 25, axis=1), np.flipud(gray)], axis=2)


def show_images(images, titles, heading, columns=3, description=None):
    rows = (len(images) + columns - 1) // columns
    figure, axes = plt.subplots(rows, columns, figsize=(4.5 * columns, 3.8 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis, image, title in zip(axes, images, titles):
        image = np.asarray(image)
        display_image = np.clip(image, 0, 255).astype(np.uint8)
        if display_image.ndim == 2:
            axis.imshow(display_image, cmap='gray', vmin=0, vmax=255)
        else:
            axis.imshow(display_image)
        axis.set_title(title, fontsize=11, fontweight='bold')
        axis.axis('off')
    for axis in axes[len(images):]:
        axis.axis('off')
    figure.suptitle(heading, fontsize=15, fontweight='bold')
    if description:
        figure.text(0.5, 0.01, description, ha='center', va='bottom', fontsize=9, wrap=True)
    figure.tight_layout(rect=(0, 0.05 if description else 0.02, 1, 0.93))
    plt.show()


print('Sample images created automatically:')
print('Grayscale:', gray.shape, '| Color:', color.shape)
show_images(
    [gray, color],
    ['Generated grayscale input', 'Generated RGB input'],
    'Automatically Generated Inputs',
    columns=2,
    description='No external image or upload is used. These two generated arrays feed every section below.',
)

## 1. Read, display, and inspect images
Input: `gray` and `color`. Output: width, height, channels, dtype, and bytes.

In [ ]:
def inspect(image):
    channels = 1 if image.ndim == 2 else image.shape[2]
    return {'width': image.shape[1], 'height': image.shape[0], 'channels': channels, 'dtype': str(image.dtype), 'bytes': image.nbytes}

print('Grayscale:', inspect(gray))
print('Color:', inspect(color))
show_images(
    [gray, color],
    ['Input: generated grayscale', 'Input: generated RGB color'],
    '1. Image Inspection',
    columns=2,
    description='Both images are generated in the setup cell; no file is read from disk.',
)

## 2. Convert color to grayscale
Input: RGB `color`. Output: one-channel grayscale image and reduced byte size.

In [ ]:
gray_from_color = np.dot(color[..., :3], [0.299, 0.587, 0.114]).astype(np.uint8)
print('Before:', color.shape, color.nbytes, 'bytes')
print('After :', gray_from_color.shape, gray_from_color.nbytes, 'bytes')
show_images(
    [color, gray_from_color],
    ['Before: generated RGB color', 'After: converted grayscale'],
    '2. Color to Grayscale Conversion',
    columns=2,
    description='The same generated color image is converted from 3 channels to 1 channel.',
)

## 3. Automatically classify the generated images
Input: the generated binary, grayscale, and RGB arrays. Output: binary, grayscale, or full-color plus reasoning.

In [ ]:
binary = np.where(gray >= 128, 255, 0).astype(np.uint8)
def classify(image):
    if image.ndim == 2:
        n = len(np.unique(image))
        return ('binary', f'detected {n} unique values') if n <= 2 else ('grayscale', f'detected {n} intensity values')
    return 'full-color', f'detected {image.shape[2]} channels'
for name, image in [('binary', binary), ('gray', gray), ('color', color)]: print(name, '=>', classify(image))

## 4. Five illustrative imaging modalities
The notebook derives five educational modality-style images from the generated sample. These are demonstrations, not external modality files.

In [ ]:
rng = np.random.default_rng(1)
modalities = [
    gray,
    color,
    np.array(Image.fromarray(gray).filter(ImageFilter.GaussianBlur(2))),
    np.clip(gray.astype(float) + rng.normal(0, 22, gray.shape), 0, 255),
    np.stack([gray, gray // 2, gray // 5], axis=2),
]
modality_names = [
    'X-ray style\n(from grayscale)',
    'Satellite style\n(from RGB)',
    'Microscopy style\n(blurred)',
    'Ultrasound style\n(noisy)',
    'Infrared style\n(channel mix)',
]
applications = ['bone-fracture diagnosis', 'land-cover mapping', 'cell inspection', 'fetal examination', 'thermal building inspection']
show_images(
    modalities,
    modality_names,
    '4. Generated Modality-Style Images',
    columns=3,
    description='Each image is derived automatically from the generated input; these are educational illustrations.',
)
for name, app in zip(['X-ray', 'Satellite', 'Microscopy', 'Ultrasound', 'Infrared'], applications):
    print(name, '-> application:', app)

## 5. Thresholding
Input: 8-bit grayscale image and threshold values 64, 128, and 192. Output: black/white images.

In [ ]:
def threshold(image, t):
    return np.where(image >= t, 255, 0).astype(np.uint8)

thresholds = [64, 128, 192]
show_images(
    [gray] + [threshold(gray, t) for t in thresholds],
    ['Before: grayscale input'] + [f'After: threshold = {t}' for t in thresholds],
    '5. Thresholding: Before and After',
    columns=2,
    description='Pixels greater than or equal to each threshold become white; the rest become black.',
)

## 6. Spatial resolution and checkerboard/block effect
Input: grayscale image. Output: 128x128, 64x64, 32x32, and 16x16 versions. As resolution decreases, pixels become larger blocks and detail disappears.

In [ ]:
sizes = [128, 64, 32, 16]
reduced_images = [np.array(Image.fromarray(gray).resize((size, size), Image.Resampling.NEAREST)) for size in sizes]
show_images(
    [gray] + reduced_images,
    ['Before: 256 x 256 input'] + [f'After: {size} x {size}' for size in sizes],
    '6. Spatial Resolution: Before and After',
    columns=3,
    description='The image becomes smaller, so each displayed pixel represents a larger block and detail is lost.',
)

## 7. False contouring / intensity resolution
Input: same spatial size, fewer gray levels. Output: 128, 64, 32, 16, 8, 4, and 2 levels. Smooth areas show visible bands.

In [ ]:
def reduce_levels(image, levels):
    return (np.floor(image / (256 / levels)) * (256 / levels)).astype(np.uint8)

levels = [128, 64, 32, 16, 8, 4, 2]
show_images(
    [gray] + [reduce_levels(gray, level) for level in levels],
    ['Before: full gray range'] + [f'After: {level} gray levels' for level in levels],
    '7. False Contouring: Before and After',
    columns=4,
    description='Reducing intensity levels creates visible bands in areas that were previously smooth.',
)

## 8. Interpolation comparison
Input: 64x64 image. Output: 4-times enlarged versions using nearest, bilinear, and bicubic interpolation.

In [ ]:
small = np.array(Image.fromarray(gray).resize((64, 64), Image.Resampling.BICUBIC))
methods = [('Nearest', Image.Resampling.NEAREST), ('Bilinear', Image.Resampling.BILINEAR), ('Bicubic', Image.Resampling.BICUBIC)]
interpolated = [np.array(Image.fromarray(small).resize((256, 256), method)) for _, method in methods]
show_images(
    [small] + interpolated,
    ['Before: 64 x 64 image'] + [f'After: {name}' for name, _ in methods],
    '8. Interpolation: Before and After',
    columns=2,
    description='The same small image is enlarged three ways: nearest is blocky, while bilinear and bicubic are smoother.',
)

## 9. Histograms
Input: dark, bright, and low-contrast versions. Output: histogram bars and interpretation.

In [ ]:
histogram_images = {
    'Dark': (gray * 0.35).astype(np.uint8),
    'Bright': np.clip(gray * 0.65 + 90, 0, 255).astype(np.uint8),
    'Low contrast': np.clip(gray * 0.18 + 95, 0, 255).astype(np.uint8),
}

figure, axes = plt.subplots(2, 3, figsize=(15, 8))
for column, (name, image) in enumerate(histogram_images.items()):
    axes[0, column].imshow(image, cmap='gray', vmin=0, vmax=255)
    axes[0, column].set_title(f'Changed image: {name}')
    axes[0, column].axis('off')
    axes[1, column].hist(image.ravel(), bins=256, range=(0, 255), color='black')
    axes[1, column].set_title(f'Histogram: {name}')
    axes[1, column].set_xlabel('Pixel intensity (0-255)')
    axes[1, column].set_ylabel('Pixel count')
figure.suptitle('9. Histogram: Changed Image and Its Distribution', fontsize=15, fontweight='bold')
figure.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()
print('Dark shifts left, bright shifts right, and low contrast becomes narrow.')

## 10. Pixel neighbors and distance measures
Input: pixel coordinate and image size. Output: N4, diagonal ND, N8, Euclidean, D4, and D8 distances.

In [ ]:
def neighbors(x, y, width, height):
    n4 = {(x-1,y),(x+1,y),(x,y-1),(x,y+1)}
    nd = {(x-1,y-1),(x-1,y+1),(x+1,y-1),(x+1,y+1)}
    valid = lambda s: {(a,b) for a,b in s if 0 <= a < width and 0 <= b < height}
    n4, nd = valid(n4), valid(nd)
    return n4, nd, n4 | nd
for point in [(128,128), (0,128), (0,0)]: print(point, neighbors(*point, 256, 256))
def distances(p, q):
    dx, dy = abs(p[0]-q[0]), abs(p[1]-q[1])
    return (np.hypot(dx,dy), dx+dy, max(dx,dy))
for p, q in [((1,2),(4,6)), ((0,0),(3,4))]: print(p, q, '=> Euclidean, D4, D8 =', distances(p,q))
print('Hand check: (1,2) to (4,6): dx=3, dy=4, so Euclidean=5, D4=7, D8=4.')

## 11. Image arithmetic and change detection
Input: two same-size images. Output: addition, subtraction, multiplication, division, and a highlighted changed region.

In [ ]:
image_a = gray.astype(float)
image_b = gray.copy()
image_b[105:145, 105:145] = 255
addition = np.clip(image_a + image_b, 0, 255)
subtraction = np.abs(image_a - image_b)
multiplication = np.clip(image_a * image_b / 255, 0, 255)
division = np.clip(image_a / np.maximum(image_b, 1) * 255, 0, 255)
results = [addition, subtraction, multiplication, division]
show_images(
    [gray, image_b] + results,
    ['Before: image A', 'Changed image B\n(white square added)', 'After: addition', 'After: subtraction\n(change map)', 'After: multiplication', 'After: division'],
    '11. Arithmetic and Change Detection',
    columns=3,
    description='Image B differs from A by one bright square. The subtraction result makes that changed area visible.',
)

## 12. Noise reduction by averaging 20 images
Input: one clean image and Gaussian noise. Output: one noisy copy and the average of 20 noisy copies.

In [ ]:
noisy_images = [np.clip(gray.astype(float) + rng.normal(0, 25, gray.shape), 0, 255) for _ in range(20)]
average = np.mean(noisy_images, axis=0)
show_images(
    [gray, noisy_images[0], average],
    ['Before: clean generated image', 'Changed: one noisy copy', 'After: average of 20 copies'],
    '12. Noise Reduction by Averaging',
    columns=3,
    description='Random noise is reduced because positive and negative errors cancel during averaging.',
)

## 13. Binary set and logical operations
Input: binary circle and square. Output: AND, OR, NOT, and XOR.

In [ ]:
circle_image = Image.new('L', (256, 256), 0)
ImageDraw.Draw(circle_image).ellipse((35, 60, 155, 180), fill=255)
a = np.array(circle_image)
square_image = Image.new('L', (256, 256), 0)
ImageDraw.Draw(square_image).rectangle((105, 85, 220, 200), fill=255)
b = np.array(square_image)
logic = [a & b, a | b, 255 - a, a ^ b]
names = ['AND = overlap', 'OR = union', 'NOT A = inverse', 'XOR = non-overlap']
show_images(
    [a, b] + logic,
    ['Input A: circle', 'Input B: square'] + [f'After: {name}' for name in names],
    '13. Logical Operations: Inputs and Results',
    columns=3,
    description='The circle and square are combined pixel by pixel using Boolean operations.',
)

## 14. Geometric transformations
Input: image, translation (25,15), rotation (25 degrees), and scale (1.35). Output: transformed images. Black areas have no source pixel.

In [ ]:
original = Image.fromarray(gray)
translated = Image.new('L', original.size, 0)
translated.paste(original, (25, 15))
rotated = original.rotate(25, fillcolor=0)
scaled = original.resize((int(w * 1.35), int(h * 1.35)))
transformed = [gray, np.array(translated), np.array(rotated), np.array(scaled)]
names = ['Before: original', 'After: translation (+25, +15)', 'After: rotation (+25 degrees)', 'After: scaling (1.35 x)']
show_images(
    transformed,
    names,
    '14. Geometric Transformations: Before and After',
    columns=2,
    description='Black areas are empty because no source pixel was present there after the transformation.',
)
print('All requested exercises are complete.')